# YOLOv26 Ablation Study: ITA-Based L*-CLAHE

This notebook runs a fair five-model comparison for the thesis defense. All models use the **same leakage-safe 70% train / 20% validation / 10% held-out test split**, the same labels, model checkpoint, image size, training budget, and seeds. They also start from the **same standardized lossless PNG pixels**. Only the preprocessing is different.

| Model | Preprocessing | Purpose |
|---|---|---|
| A | Standardized images, no enhancement | Raw baseline |
| B | Per-channel RGB CLAHE, fixed β=2.0 | Standard enhancement baseline |
| C | CIELAB L*-CLAHE, fixed β=2.0 | Fixed perceptual-space baseline |
| C′ | CIELAB L*-CLAHE, fixed β = `beta_global` | **Control:** same calibration procedure as D on all images pooled, with no ITA |
| D | Dynamic K-means mask + ITA-based L*-CLAHE (β_high / β_mid / β_low) | Proposed framework |

**Why C′ exists:** if D beats C, the panel can ask whether the gain came from the ITA logic or only from a stronger clip limit. C′ uses the same calibration procedure but ignores skin tone, so **D vs C′ isolates the effect of the ITA-based β selection.**

> Do not tune anything using the test set. Set every parameter before training, select `best.pt` by validation mAP, then evaluate each model on the held-out test set exactly once.

## Prerequisites

Run these once from the `TOOL-26` folder, in order. Select the **TOOL-26 (.venv)** kernel for this notebook.

```bash
# 0. Source manifest from the team archive (1. Filtered + 2. Annotated, verified pairs only)
python backend/build_team_source_manifest.py \
  --dataset-root "phase0_work/team_dataset/Data Set" \
  --source-repository "TEAM_DATASET:Data Set-20260925T122828Z-1-001.zip" \
  --manual-exclusions docs/team_dataset_manual_exclusions.csv \
  --output phase0_work/team_inventory/team_source_manifest.csv

# 1. Quality + duplicate removal + leakage-safe 70/20/10 split
python backend/run_member1_batch.py prepare-split-manifest --quality-scope calibration \
  --input-manifest phase0_work/team_inventory/team_source_manifest.csv \
  --output-dir phase0_work/team_prepared

# 2. Phase 0 masking + ITA on the TRAIN split only
python backend/run_member1_batch.py run-phase0 \
  --manifest phase0_work/team_prepared/cleaned_split_manifest.csv \
  --output-dir phase0_outputs/member1_team_train --allow-provisional

# 3. Beta calibration (writes beta_high/mid/low and beta_global)
python backend/clahe_calibration.py \
  phase0_outputs/member1_team_train/image_ita_manifest.csv \
  --output phase0_outputs/member2_calibration
cp phase0_outputs/member2_calibration/phase0_calibration.json backend/phase0_calibration.json

# 4. YOLO dataset from the same split
python backend/build_yolo_dataset.py \
  --manifest phase0_work/team_prepared/cleaned_split_manifest.csv \
  --output datasets/source_yolo
```

`CLASS_NAMES` is read from `datasets/source_yolo/classes.json`. Classes without enough reliably matched annotations are excluded there, and the exclusion is logged.

In [ ]:
from __future__ import annotations

import json
import shutil
import sys
import warnings
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'backend').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / 'backend').is_dir(), 'Open this notebook from TOOL-26/notebooks or TOOL-26.'

sys.path.insert(0, str(PROJECT_ROOT / 'backend'))
from masking_ita import MaskingITAConfig, MaskingITAProcessor
from clahe_calibration import apply_clahe, ita_to_beta

DATASET_SOURCE = PROJECT_ROOT / 'datasets' / 'source_yolo'
ABLATION_ROOT = PROJECT_ROOT / 'datasets' / 'ablation_yolov26'
RUNS_ROOT = PROJECT_ROOT / 'runs' / 'ablation_yolov26'
WEIGHTS_ROOT = PROJECT_ROOT / 'weights' / 'ablation_yolov26'
PHASE0_ITA_MANIFEST = PROJECT_ROOT / 'phase0_outputs' / 'member1_team_train' / 'image_ita_manifest.csv'
CALIBRATION_JSON = PROJECT_ROOT / 'backend' / 'phase0_calibration.json'
MEMBER1_CONFIG_JSON = PROJECT_ROOT / 'backend' / 'member1_phase0_config.json'

classes_info = json.loads((DATASET_SOURCE / 'classes.json').read_text(encoding='utf-8'))
CLASS_NAMES = classes_info['classes']
if classes_info['excluded_classes_train_label_counts']:
    warnings.warn(f"Classes excluded for missing annotations: {classes_info['excluded_classes_train_label_counts']}")

MODEL_CHECKPOINT = 'yolo26n.pt'  # Same checkpoint for every model.
SEEDS = (42,)                    # One seed to finish overnight; add 43 and 44 later for mean ± SD (finished runs are skipped).
PRIMARY_SEED = SEEDS[0]          # Used for plots and visual proofs.
IMG_SIZE = 640
EPOCHS = 300
PATIENCE = 50
BATCH = 8                        # Fits a 4 GB GPU; keep identical for every model.
FIXED_BETA = 2.0
TILE_GRID_SIZE = (8, 8)

calibration = json.loads(CALIBRATION_JSON.read_text(encoding='utf-8'))
BETA_HIGH = float(calibration['beta_high'])
BETA_MID = float(calibration['beta_mid'])
BETA_LOW = float(calibration['beta_low'])
assert calibration.get('beta_global') is not None, 'Re-run clahe_calibration.py: beta_global is required for Model C′.'
BETA_GLOBAL = float(calibration['beta_global'])

if calibration.get('status') != 'FROZEN':
    warnings.warn('phase0_calibration.json is not FROZEN. Results are provisional until the team approves the calibration.')
if len({BETA_HIGH, BETA_MID, BETA_LOW}) == 1:
    warnings.warn('beta_high == beta_mid == beta_low: Model D is then identical to a fixed-beta model and cannot demonstrate the ITA effect.')

print({'classes': CLASS_NAMES, 'betas': {'high': BETA_HIGH, 'mid': BETA_MID, 'low': BETA_LOW, 'global': BETA_GLOBAL},
       'selection_rule': calibration['beta_search'].get('selection_rule'), 'status': calibration.get('status')})

In [ ]:
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
SPLITS = ('train', 'val', 'test')

def image_files(split: str, root: Path = DATASET_SOURCE):
    return sorted(p for p in (root / 'images' / split).rglob('*') if p.suffix.lower() in IMAGE_SUFFIXES)

def label_for(image_path: Path, split: str, root: Path = DATASET_SOURCE) -> Path:
    relative = image_path.relative_to(root / 'images' / split).with_suffix('.txt')
    return root / 'labels' / split / relative

assert DATASET_SOURCE.is_dir(), f'Missing dataset: {DATASET_SOURCE}'
counts = {}
for split in SPLITS:
    images = image_files(split)
    assert images, f'No images in {split}'
    for image in images:
        label = label_for(image, split)
        assert label.is_file(), f'Missing label: {label}'
        for line in label.read_text().splitlines():
            class_id = int(line.split()[0])
            assert 0 <= class_id < len(CLASS_NAMES), f'Bad class id {class_id} in {label}'
    counts[split] = len(images)

total = sum(counts.values())
shares = {key: round(value / total, 3) for key, value in counts.items()}
print('Image counts:', counts, 'shares:', shares)
assert 0.65 <= shares['train'] <= 0.75 and 0.15 <= shares['val'] <= 0.25 and 0.05 <= shares['test'] <= 0.15, 'Expected a 70/20/10 split.'

## ITA for every image

Model D needs an ITA for every image it sees. Training images reuse the Phase 0 result, which comes from the same processor, configuration, and pixels. Validation and test images are processed here as ordinary inference and are **never** used for calibration. An image whose mask fails (`MASK_FAILED`) has no ITA. Model D then uses the ITA-agnostic `beta_global`, and the number of these fallbacks is reported.

In [ ]:
member1_config = json.loads(MEMBER1_CONFIG_JSON.read_text(encoding='utf-8'))
processor = MaskingITAProcessor(MaskingITAConfig(**member1_config['masking']))
ITA_CACHE = ABLATION_ROOT / 'ita_table.csv'

def build_ita_table() -> pd.DataFrame:
    if ITA_CACHE.is_file():
        return pd.read_csv(ITA_CACHE)
    phase0 = pd.read_csv(PHASE0_ITA_MANIFEST).set_index('image_id') if PHASE0_ITA_MANIFEST.is_file() else None
    if phase0 is None:
        warnings.warn('Phase 0 manifest not found; ITA will be computed for training images too.')
    rows = []
    for split in SPLITS:
        images = image_files(split)
        for index, path in enumerate(images, start=1):
            image_id = path.stem
            if split == 'train' and phase0 is not None and image_id in phase0.index:
                row = phase0.loc[image_id]
                ita = row['ITA'] if pd.notna(row['ITA']) else np.nan
                rows.append({'split': split, 'image_id': image_id, 'ita': ita,
                             'bracket': row['bracket'] if pd.notna(ita) else '', 'status': row['status'], 'source': 'phase0'})
            else:
                result = processor.process_image(processor.load_image(path), filename=path.name)
                rows.append({'split': split, 'image_id': image_id,
                             'ita': result.ita if result.ita is not None else np.nan,
                             'bracket': result.bracket or '', 'status': result.status, 'source': 'computed'})
            if index % 50 == 0 or index == len(images):
                print(f'{split}: {index}/{len(images)}')
    table = pd.DataFrame(rows)
    ITA_CACHE.parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(ITA_CACHE, index=False)
    return table

ita_table = build_ita_table()
ITA_BY_ID = ita_table.set_index('image_id')
print(pd.crosstab(ita_table['split'], ita_table['bracket'].replace('', 'no ITA (MASK_FAILED)'), margins=True))

## Build the five preprocessed datasets

Every model reads the same standardized image through the same loader and writes lossless PNG, so neither compression nor color management can confound the comparison.

In [ ]:
MODELS = ('ModelA_raw', 'ModelB_rgb_clahe', 'ModelC_fixed_l_clahe', 'ModelC2_fixed_l_clahe_global', 'ModelD_proposed')

def rgb_clahe(image_rgb: np.ndarray) -> np.ndarray:
    clahe = cv2.createCLAHE(clipLimit=FIXED_BETA, tileGridSize=TILE_GRID_SIZE)
    return cv2.merge([clahe.apply(channel) for channel in cv2.split(image_rgb)])

def model_d_beta(image_id: str) -> tuple[float, dict]:
    row = ITA_BY_ID.loc[image_id]
    if pd.isna(row['ita']):
        return BETA_GLOBAL, {'ita': np.nan, 'bracket': '', 'status': row['status'], 'fallback': True}
    beta = ita_to_beta(float(row['ita']), BETA_HIGH, BETA_MID, BETA_LOW)
    return beta, {'ita': float(row['ita']), 'bracket': row['bracket'], 'status': row['status'], 'fallback': False}

def transform(model_name: str, rgb: np.ndarray, image_id: str) -> tuple[np.ndarray, dict]:
    if model_name == 'ModelA_raw':
        return rgb, {}
    if model_name == 'ModelB_rgb_clahe':
        return rgb_clahe(rgb), {'beta': FIXED_BETA}
    if model_name == 'ModelC_fixed_l_clahe':
        return apply_clahe(rgb, beta=FIXED_BETA, tile_grid_size=TILE_GRID_SIZE), {'beta': FIXED_BETA}
    if model_name == 'ModelC2_fixed_l_clahe_global':
        return apply_clahe(rgb, beta=BETA_GLOBAL, tile_grid_size=TILE_GRID_SIZE), {'beta': BETA_GLOBAL}
    if model_name == 'ModelD_proposed':
        beta, details = model_d_beta(image_id)
        return apply_clahe(rgb, beta=beta, tile_grid_size=TILE_GRID_SIZE), {'beta': beta, **details}
    raise ValueError(model_name)

def make_dataset(model_name: str) -> Path:
    target = ABLATION_ROOT / model_name
    audit_path = target / 'preprocessing_audit.csv'
    if audit_path.is_file():
        print(f'{model_name}: already prepared, delete {target} to rebuild.')
        return target
    audit = []
    for split in SPLITS:
        for source_image in image_files(split):
            rel = source_image.relative_to(DATASET_SOURCE / 'images' / split)
            destination_image = (target / 'images' / split / rel).with_suffix('.png')
            destination_label = target / 'labels' / split / rel.with_suffix('.txt')
            destination_image.parent.mkdir(parents=True, exist_ok=True)
            destination_label.parent.mkdir(parents=True, exist_ok=True)
            rgb = processor.load_image(source_image)
            transformed, details = transform(model_name, rgb, source_image.stem)
            ok = cv2.imwrite(str(destination_image), cv2.cvtColor(transformed, cv2.COLOR_RGB2BGR))
            assert ok, f'Could not write {destination_image}'
            shutil.copy2(label_for(source_image, split), destination_label)
            audit.append({'split': split, 'image': str(rel), **details})
    pd.DataFrame(audit).to_csv(audit_path, index=False)
    return target

for model_name in MODELS:
    print(f'Preparing {model_name}...')
    make_dataset(model_name)

audit_d = pd.read_csv(ABLATION_ROOT / 'ModelD_proposed' / 'preprocessing_audit.csv')
print('\nModel D beta usage per split:')
print(pd.crosstab(audit_d['split'], audit_d['beta']))
print('Model D ITA fallbacks (MASK_FAILED -> beta_global):', audit_d.groupby('split')['fallback'].sum().to_dict())

In [ ]:
BRACKETS = ('Darkest', 'Medium', 'Lightest')

def write_data_yaml(model_name: str, test_list: Path | None = None, suffix: str = '') -> Path:
    root = ABLATION_ROOT / model_name
    payload = {
        'path': str(root),
        'train': 'images/train',
        'val': 'images/val',
        'test': str(test_list) if test_list else 'images/test',
        'names': {index: name for index, name in enumerate(CLASS_NAMES)},
    }
    yaml_path = root / f'data{suffix}.yaml'
    yaml_path.write_text(yaml.safe_dump(payload, sort_keys=False), encoding='utf-8')
    return yaml_path

DATA_YAMLS = {name: write_data_yaml(name) for name in MODELS}

# Per-skin-tone test subsets: the bracket comes from the ITA of the unenhanced test image.
BRACKET_YAMLS = {}
test_ita = ita_table[ita_table['split'] == 'test']
for name in MODELS:
    for bracket in BRACKETS:
        ids = test_ita.loc[test_ita['bracket'] == bracket, 'image_id']
        if ids.empty:
            continue
        list_path = ABLATION_ROOT / name / f'test_{bracket}.txt'
        list_path.write_text('\n'.join(str(ABLATION_ROOT / name / 'images' / 'test' / f'{i}.png') for i in ids) + '\n')
        BRACKET_YAMLS[(name, bracket)] = write_data_yaml(name, list_path, suffix=f'_test_{bracket}')

print('Test images per ITA bracket:', test_ita['bracket'].replace('', 'no ITA').value_counts().to_dict())
DATA_YAMLS

## Train every model × seed

The validation split is checked every epoch. `patience=50` stops a run after 50 epochs without improvement in validation mAP, and Ultralytics saves the validation-selected weights as `best.pt`.

**Training one model at a time.** Set `TRAIN_ONLY` below, for example `('ModelA_raw',)`, run this cell, and move on to the next model in a later session. The order does not matter.

**If something goes wrong** (power loss, crash, closed window), just run the cells again:
- Finished runs are skipped.
- An interrupted run **resumes from its last saved epoch** (`last.pt`) instead of starting over.

Every model must be trained on the same machine with the same settings. Evaluation on the test set runs only after all `len(MODELS) × len(SEEDS)` runs exist, so the test set is used once, for all models together.

In [ ]:
import torch

TRAIN_ONLY = ('ModelD_proposed',)  # D first (needed for the app); then set None to train the rest.

WEIGHTS_ROOT.mkdir(parents=True, exist_ok=True)

def run_dir(model_name: str, seed: int) -> Path:
    return RUNS_ROOT / 'training' / f'{model_name}_seed{seed}'

def checkpoint_state(checkpoint: Path) -> str:
    # Returns 'finished', 'resumable', or 'corrupt' (e.g. power loss while saving).
    try:
        epoch = torch.load(checkpoint, map_location='cpu', weights_only=False).get('epoch', 0)
    except Exception:
        return 'corrupt'
    # Ultralytics sets epoch = -1 in the checkpoint once training has completed.
    return 'finished' if epoch == -1 else 'resumable'

def train_one(model_name: str, seed: int) -> Path:
    destination = WEIGHTS_ROOT / f'best_{model_name}_seed{seed}.pt'
    if destination.is_file():
        return destination
    weights_dir = run_dir(model_name, seed) / 'weights'
    last = weights_dir / 'last.pt'
    state = checkpoint_state(last) if last.is_file() else 'missing'
    if state == 'corrupt':
        print(f'{last} is unreadable; restarting {model_name} seed {seed} from scratch.')
        shutil.rmtree(run_dir(model_name, seed))
        state = 'missing'
    if state == 'resumable':
        print(f'Resuming {model_name} seed {seed} from {last}')
        YOLO(str(last)).train(resume=True)
    elif state == 'missing':
        YOLO(MODEL_CHECKPOINT).train(
            data=str(DATA_YAMLS[model_name]), epochs=EPOCHS, patience=PATIENCE,
            imgsz=IMG_SIZE, batch=BATCH, seed=seed, deterministic=True,
            project=str(RUNS_ROOT / 'training'), name=f'{model_name}_seed{seed}', exist_ok=True,
            plots=True, verbose=True,
        )
    best = weights_dir / 'best.pt'
    assert best.is_file(), f'Missing validation-selected weights: {best}'
    shutil.copy2(best, destination)
    return destination

for model_name in (TRAIN_ONLY or MODELS):
    assert model_name in MODELS, f'Unknown model: {model_name}'
    for seed in SEEDS:
        train_one(model_name, seed)

best_weights = {
    (model_name, seed): WEIGHTS_ROOT / f'best_{model_name}_seed{seed}.pt'
    for model_name in MODELS for seed in SEEDS
    if (WEIGHTS_ROOT / f'best_{model_name}_seed{seed}.pt').is_file()
}
missing = [f'{m} seed {s}' for m in MODELS for s in SEEDS if (m, s) not in best_weights]
print('Finished runs:', len(best_weights), '/', len(MODELS) * len(SEEDS))
print('Still to train:', missing or 'none')

# Stable names for the primary seed: best_ModelA_raw.pt ... best_ModelD_proposed.pt
for model_name in MODELS:
    if (model_name, PRIMARY_SEED) in best_weights:
        shutil.copy2(best_weights[(model_name, PRIMARY_SEED)], WEIGHTS_ROOT / f'best_{model_name}.pt')
sorted(p.name for p in WEIGHTS_ROOT.glob('*.pt'))

In [ ]:
# Training curves (primary seed): box_loss and cls_loss should keep falling; validation mAP should plateau.
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for model_name in MODELS:
    results_csv = run_dir(model_name, PRIMARY_SEED) / 'results.csv'
    if not results_csv.is_file():
        continue
    history = pd.read_csv(results_csv)
    history.columns = history.columns.str.strip()
    axes[0].plot(history['epoch'], history['val/box_loss'], label=model_name)
    axes[1].plot(history['epoch'], history['val/cls_loss'], label=model_name)
    axes[2].plot(history['epoch'], history['metrics/mAP50-95(B)'], label=model_name)
for axis, title in zip(axes, ('Validation box_loss', 'Validation cls_loss', 'Validation mAP@0.5:0.95')):
    axis.set_title(title); axis.set_xlabel('epoch'); axis.grid(alpha=0.3)
axes[2].legend(fontsize=8)
plt.tight_layout()
(RUNS_ROOT / 'figures').mkdir(parents=True, exist_ok=True)
plt.savefig(RUNS_ROOT / 'figures' / 'training_curves.png', dpi=200, bbox_inches='tight')

## Final evaluation on the untouched 10% test split

Do not resume training or change any setting after this point. These are the final, reportable test scores.

In [ ]:
def evaluate(weights: Path, data_yaml: Path, name: str):
    results = YOLO(str(weights)).val(
        data=str(data_yaml), split='test', imgsz=IMG_SIZE, batch=BATCH,
        project=str(RUNS_ROOT / 'evaluation'), name=name, exist_ok=True, plots=True, verbose=False,
    )
    return results, {'mAP@0.5': results.box.map50, 'mAP@0.5:0.95': results.box.map,
                     'precision': results.box.mp, 'recall': results.box.mr}

assert not missing, f'Train every model before touching the test set. Still missing: {missing}'

metrics_rows, evaluation_dirs = [], {}
for (model_name, seed), weights in best_weights.items():
    results, scores = evaluate(weights, DATA_YAMLS[model_name], f'{model_name}_seed{seed}')
    evaluation_dirs[(model_name, seed)] = Path(results.save_dir)
    metrics_rows.append({'model': model_name, 'seed': seed, **scores})

per_seed = pd.DataFrame(metrics_rows)
per_seed.to_csv(RUNS_ROOT / 'ablation_test_metrics_per_seed.csv', index=False)
summary = per_seed.groupby('model')[['mAP@0.5', 'mAP@0.5:0.95', 'precision', 'recall']].agg(['mean', 'std'])
summary = summary.loc[list(MODELS)]
summary.to_csv(RUNS_ROOT / 'ablation_test_metrics.csv')
summary.style.format('{:.3f}')

In [ ]:
# Per-skin-tone evaluation: the most direct evidence for the ITA logic.
bracket_rows = []
for (model_name, bracket), data_yaml in BRACKET_YAMLS.items():
    for seed in SEEDS:
        _, scores = evaluate(best_weights[(model_name, seed)], data_yaml, f'{model_name}_seed{seed}_{bracket}')
        bracket_rows.append({'model': model_name, 'bracket': bracket, 'seed': seed, **scores})

per_bracket = pd.DataFrame(bracket_rows)
per_bracket.to_csv(RUNS_ROOT / 'ablation_test_metrics_by_bracket_per_seed.csv', index=False)
bracket_table = per_bracket.pivot_table(index='model', columns='bracket', values='mAP@0.5:0.95', aggfunc=['mean', 'std'])
bracket_table = bracket_table.loc[list(MODELS)]
bracket_table.to_csv(RUNS_ROOT / 'ablation_test_map50_95_by_bracket.csv')
print('Test images per bracket:', test_ita['bracket'].value_counts().to_dict(), '(small subsets: interpret with care)')
bracket_table.style.format('{:.3f}')

In [ ]:
# YOLO writes the confusion matrix, PR curve, F1 curve, and related plots in each evaluation folder.
# Copy the primary-seed plots into one folder for the manuscript and slides.
figures = RUNS_ROOT / 'figures'
for model_name in MODELS:
    folder = evaluation_dirs[(model_name, PRIMARY_SEED)]
    for plot in sorted(folder.glob('*.png')):
        if 'batch' in plot.name:
            continue
        shutil.copy2(plot, figures / f'{model_name}_{plot.name}')
sorted(p.name for p in figures.glob('*.png'))

## Visual proof: Model A vs Model D

Two held-out test images per class. Each model receives **its own** preprocessed version of the image, because Model D was trained on enhanced images. Feeding it raw images would understate its performance.

In [ ]:
def draw_ground_truth(model_name: str, image_id: str) -> np.ndarray:
    image = cv2.imread(str(ABLATION_ROOT / model_name / 'images' / 'test' / f'{image_id}.png'))
    height, width = image.shape[:2]
    for line in (ABLATION_ROOT / model_name / 'labels' / 'test' / f'{image_id}.txt').read_text().splitlines():
        class_id, x, y, w, h = line.split()
        x, y, w, h = float(x) * width, float(y) * height, float(w) * width, float(h) * height
        cv2.rectangle(image, (int(x - w / 2), int(y - h / 2)), (int(x + w / 2), int(y + h / 2)), (0, 200, 0), max(2, width // 250))
        cv2.putText(image, CLASS_NAMES[int(class_id)], (int(x - w / 2), max(15, int(y - h / 2) - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, max(0.5, width / 1200), (0, 200, 0), 2)
    return image

# Two test images per class, chosen deterministically (first by image id), not by looking at predictions.
test_labels = {p.stem: int(p.read_text().split()[0]) for p in (DATASET_SOURCE / 'labels' / 'test').glob('*.txt')}
comparison_ids = []
for class_id in range(len(CLASS_NAMES)):
    comparison_ids += sorted(i for i, c in test_labels.items() if c == class_id)[:2]

columns = ('Ground truth', 'ModelA_raw', 'ModelD_proposed')
fig, axes = plt.subplots(len(comparison_ids), len(columns), figsize=(15, 4.5 * len(comparison_ids)))
visual_root = RUNS_ROOT / 'visual_proof'
visual_root.mkdir(parents=True, exist_ok=True)
models = {name: YOLO(str(best_weights[(name, PRIMARY_SEED)])) for name in columns[1:]}
for row, image_id in enumerate(comparison_ids):
    for col, name in enumerate(columns):
        if name == 'Ground truth':
            rendered = draw_ground_truth('ModelA_raw', image_id)
        else:
            source = ABLATION_ROOT / name / 'images' / 'test' / f'{image_id}.png'
            rendered = models[name].predict(str(source), imgsz=IMG_SIZE, conf=0.25, verbose=False)[0].plot()
            cv2.imwrite(str(visual_root / f'{name}_{image_id}.png'), rendered)
        axes[row, col].imshow(cv2.cvtColor(rendered, cv2.COLOR_BGR2RGB))
        bracket = ITA_BY_ID.loc[image_id, 'bracket'] or 'no ITA'
        axes[row, col].set_title(f'{name} | {CLASS_NAMES[test_labels[image_id]]} | {bracket}', fontsize=10)
        axes[row, col].axis('off')
plt.tight_layout()
plt.savefig(visual_root / 'model_a_vs_d_side_by_side.png', dpi=150, bbox_inches='tight')

## Defense checklist

- Report the mean ± SD of `mAP@0.5` and `mAP@0.5:0.95` across seeds from `runs/ablation_yolov26/ablation_test_metrics.csv`.
- The key comparisons are **D vs C′**, which isolates the ITA-based β selection, and D vs A/B/C, which shows the whole pipeline.
- Show the per-bracket table `ablation_test_map50_95_by_bracket.csv`. The claim that dark skin benefits most must be visible in the Darkest column. State the number of images per bracket.
- Preserve `weights/ablation_yolov26/best_Model*_seed*.pt`, and the primary-seed copies `best_Model*.pt`.
- Include each model's confusion matrix and PR curve from `runs/ablation_yolov26/figures/`, plus `training_curves.png`.
- Present `runs/ablation_yolov26/visual_proof/model_a_vs_d_side_by_side.png`.
- Report the number of Model D ITA fallbacks (MASK_FAILED → `beta_global`) printed by the dataset-preparation cell.
- State that every run had the same data split, standardized source pixels, annotations, checkpoint, training budget, seeds, and early-stopping patience, so preprocessing was the only experimental variable.
- State the class coverage from `datasets/source_yolo/classes.json`, including which classes were excluded for missing annotations.
- If `phase0_calibration.json` is not `FROZEN`, call the results provisional.
- If Model D is not the best model, report that transparently. Investigate only in a new experiment, never by changing the held-out test set.